In [1]:
import sys
import warnings
warnings.filterwarnings("ignore")
sys.path.insert(0,'..')
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers.optimization import AdamW
from source.version1.data import trainLoader
from source.version1.model import XLMModel
from source.version1.train import trainModel
from source.version1.loss import WeightedLoss

In [3]:
exc = ['bias','LayerNorm.bias','LayerNorm.weight']

In [4]:
def train(load, subset, rate):
    torch.cuda.empty_cache()
    train, valid = trainLoader(subset)
    model = XLMModel()
    params = list(model.named_parameters())
    groups = []
    groups += [{'params' : [p for n,p in params if not any(ex in n for ex in exc)], 
                'weight_decay':0.01}]
    groups += [{'params' : [p for n,p in params if any(ex in n for ex in exc)], 
                'weight_decay':0.00}]
    model = model.to('cuda:0')
    optimizer = AdamW(groups, lr=rate)
    schedular = ReduceLROnPlateau(optimizer, factor=0.5, min_lr=1e-6, patience=0)
    trainer = {}
    trainer['subset'] = subset
    trainer['model'] = model
    trainer['train'] = train
    trainer['valid'] = valid
    trainer['loss_fn'] = WeightedLoss()
    trainer['optimizer'] = optimizer
    trainer['save'] = '../../model/version1/'
    trainer['epochs'] = 5
    trainer['batch'] = 8
    trainer['schedular'] = schedular
    trainModel(**trainer)
    model = model.cpu()
    del model
    torch.cuda.empty_cache()
    return None

In [ ]:
train(True, 0, 1e-5)

Data: (313661, 9)


 22%|██▏       | 11128/50000 [11:23<39:11, 16.53it/s, train_loss=0.0169]

In [ ]:
# train(True, 1, 1e-5)

In [ ]:
# train(True, 2, 1e-5)

In [ ]:
# train(True, 3, 1e-5)

In [ ]:
# train(True, 4, 1e-5)